# ЛР4. Цифровой двойник, этап 4: торможение и привод без датчика

In [ ]:
# Подготовка среды: папки scripts, autograder и detective должны лежать рядом с notebooks
# (в Colab: загрузите архив материалов дисциплины и распакуйте его в /content)
import sys, os, numpy as np, matplotlib.pyplot as plt
for p in ("../scripts", "scripts", "/content/scripts", "../detective", "/content/detective"):
    if os.path.isdir(p): sys.path.insert(0, os.path.abspath(p))
from srm_model import SRM, simulate_time, cycle_angle_domain
mot = SRM(); deg = np.deg2rad
print("Модель загружена: ВИД", f"{mot.Ns}/{mot.Nr}", "Udc =", mot.Udc, "В")

In [ ]:
# ЗАДАНИЕ: улучшите оценку (исследовательский уровень). Текст функции сохраняется для автопроверки.
ESTIMATOR = '''
import numpy as np
def estimate(La, Lb, Lc):
    l_al = (2*La - Lb - Lc)/3
    l_be = (Lb - Lc)/np.sqrt(3)
    return np.mod(np.arctan2(-l_be, -l_al), 2*np.pi)
'''
exec(ESTIMATOR)
thm = np.linspace(0, 2*np.pi/mot.Nr, 720, endpoint=False)
L = [mot.L_unsat(mot.phase_angle(thm, p)) for p in range(3)]
err = np.rad2deg(np.angle(np.exp(1j*(estimate(*L) - mot.Nr*thm))))
plt.plot(np.rad2deg(mot.Nr*thm), err); plt.xlabel('эл. град'); plt.ylabel('ошибка, эл. град'); plt.grid(alpha=.3)
print('амплитуда ошибки', round(float(np.abs(err).max()), 3), 'эл. град')

In [ ]:
import pathlib
out = pathlib.Path("results/lab4"); out.mkdir(parents=True, exist_ok=True)
(out/"lab4_estimator.py").write_text(ESTIMATOR, encoding="utf-8")

## Неисправный привод
Преподаватель выдает файл `case_XX.csv`. Определите вид неисправности и фазу. Подсказка: рассчитайте потокосцепление каждой фазы интегрированием $u - R i$ и сравните фазы между собой, оцените нулевой уровень токов и частоту переключений.

In [ ]:
import make_cases
rec = make_cases.simulate_case("turn_fault", 1, 450, 12.0, np.random.default_rng(0), periods=2)  # пример
t, ia, ib, ic, ua, ub, uc = rec[:,0], rec[:,2], rec[:,3], rec[:,4], rec[:,5], rec[:,6], rec[:,7]
dt = t[1]-t[0]
for name, i_, u_ in (("A", ia, ua), ("B", ib, ub), ("C", ic, uc)):
    psi = np.maximum(np.cumsum(u_ - mot.R*i_)*dt, 0)
    print(name, "максимум потокосцепления", round(float(psi.max()), 3), "Вб, число переключений", int(np.count_nonzero(np.diff(u_))))